# Retrieval for RAG

Adnan Masood · University of South Florida

Search four short disclosure passages for evidence about customer concentration. Retrieval chooses which passages an answer writer can inspect; it does not write or verify an answer by itself.

**Dataset:** The notebook uses four hand-written passages and one question. Passages 1 and 3 are manually marked relevant to customer concentration. Those marks are teaching labels and must be updated when the question changes; they are not judgments made by the search model.

**Task:** Rank passages with TF-IDF cosine similarity. TF-IDF gives more weight to words that help distinguish one passage from the others; cosine similarity compares their word patterns with the question. At K=2, inspect the top two passages. Retrieval recall is relevant passages found divided by all relevant passages; context precision is relevant passages found divided by passages retrieved.

**Compare:** Keep the question and relevance labels fixed while comparing the top 2 passages with the top 3. A larger K may find more relevant evidence, but it also gives the answer writer more unrelated text. Then ask for a revenue figure absent from every passage and check whether the system should answer or report that the evidence is missing.

**Runtime:** The browser uses NumPy and scikit-learn to rank word patterns; it does not generate an answer. The separate native example uses Sentence-Transformers to compare semantic text vectors and needs an initial model download. Relevance and answerability are supplied annotations, not model judgments.

[Download the native Python example](native/rag.py).


In [ ]:
# Offline TF-IDF retrieval baseline; no pretrained embedding model runs here.
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
chunks = ["No customer accounted for more than 10 percent of revenue.",
 "Interest expense increased because average debt was higher.",
 "A small number of distributors represent a material share of sales.",
 "The company maintains a revolving credit facility."]
question = "What customer concentration risk does the company disclose?"
relevant = {0, 2}
encoder = TfidfVectorizer(stop_words="english")
D = encoder.fit_transform(chunks)
q = encoder.transform([question])
scores = cosine_similarity(q,D)[0]
K = 2
top = scores.argsort()[::-1][:K]
hits = len(set(top) & relevant)
print(f"Retrieval Recall@{K}:", hits / len(relevant) if relevant else "undefined: no required passage is present")
print(f"Context precision@{K}:", hits / len(top))
for i in top: print("Passage", int(i+1), "score", round(float(scores[i]),3), chunks[i])
print("Answerability annotation:", "evidence is present" if relevant else "no supporting evidence in this collection")
print("This lab ranks passages. It does not generate an answer or automatically decide when to abstain.")


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Ask a question without evidence

No passage supplies that amount. Recall is undefined, context precision is zero, and returned passages do not establish answerability.

Replace:
```python
question = "What customer concentration risk does the company disclose?"
relevant = {0, 2}
```
With:
```python
question = "What was revenue in 2023?"
relevant = set()
```

### Retrieve three passages

Compare which passage is added and recalculate recall and context precision.

Replace:
```python
K = 2
```
With:
```python
K = 3
```



In [ ]:
# Change K from 2 to 3 and record which passages were added.
# Recalculate recall using two required passages and precision using the number retrieved.


## Break it

Ask 'What was revenue in 2023?' and set relevant=set(), because no passage provides that number. Recall is undefined when there are no required passages in the collection; context precision is zero. Retrieval still returns text, so the presence of results is not evidence that the question is answerable.


In [ ]:
# Change both question and relevant together for the absent-evidence case.
# Inspect the returned passages and explain why none supports a revenue amount.


## What to submit

- The top passages for K=2 and K=3, with recall calculated as relevant passages found divided by two and precision as relevant passages found divided by the number retrieved.
- The absent-revenue question with an empty set of relevant passages. Explain why retrieval can still return text even though no passage supports a revenue answer.
- A proposed evidence check that connects each answer claim to its source, and a clear distinction between retrieval (finding text) and generation (writing an answer).

## Optional extension

Download native/rag.py for Sentence-Transformers retrieval with all-MiniLM-L6-v2. Neither example generates an answer or implements an automatic abstention policy.
